# Crypto Market Analysis

## Problem Statement

This project investigates how historical trader activity and performance relate to Bitcoin market sentiment. Phase 1 focuses only on loading the two source datasets, understanding their schemas, and documenting data-quality observations. No values are cleaned, converted, merged, or visualized in this notebook.

## Import Libraries

In [1]:
from pathlib import Path
import sys

import pandas as pd

project_root = Path.cwd()
if not (project_root / "src").is_dir():
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.loader import (
    get_dataset_summary,
    load_fear_greed_data,
    load_historical_data,
)

## Load Datasets

The reusable loader functions resolve the configured raw-data paths and validate each expected schema.

In [2]:
historical_data = load_historical_data()
fear_greed_data = load_fear_greed_data()

print("Historical trader data loaded:", historical_data.shape)
print("Fear & Greed Index data loaded:", fear_greed_data.shape)

Historical trader data loaded: (211224, 16)
Fear & Greed Index data loaded: (2644, 4)


## Dataset Overview

In [3]:
historical_summary = get_dataset_summary(historical_data)
fear_greed_summary = get_dataset_summary(fear_greed_data)

def overview_table(summary: dict) -> pd.DataFrame:
    """Build a readable schema and memory-usage table."""
    return pd.DataFrame(
        {
            "dtype": summary["dtypes"],
            "memory_bytes": summary["memory_usage_by_column_bytes"],
        }
    )

### Historical Trader Data

In [4]:
print("Shape:", historical_summary["shape"])
print("Columns:", historical_summary["columns"])
print("Total memory usage (bytes):", historical_summary["memory_usage_bytes"])
print(overview_table(historical_summary).to_string())
print("\nFirst five rows:")
print(historical_data.head().to_string())
print("\nLast five rows:")
print(historical_data.tail().to_string())

Shape: (211224, 16)
Columns: ['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']
Total memory usage (bytes): 107159892
                    dtype  memory_bytes
Account            object      19221384
Coin               object      11192873
Execution Price   float64       1689792
Size Tokens       float64       1689792
Size USD          float64       1689792
Side               object      11092176
Timestamp IST      object      13729560
Start Position    float64       1689792
Direction          object      12213655
Closed PnL        float64       1689792
Transaction Hash   object      24290760
Order ID            int64       1689792
Crossed              bool        211224
Fee               float64       1689792
Trade ID          float64       1689792
Timestamp         float64       1689792

First five rows:
                        

### Fear & Greed Index Data

In [5]:
print("Shape:", fear_greed_summary["shape"])
print("Columns:", fear_greed_summary["columns"])
print("Total memory usage (bytes):", fear_greed_summary["memory_usage_bytes"])
print(overview_table(fear_greed_summary).to_string())
print("\nFirst five rows:")
print(fear_greed_data.head().to_string())
print("\nLast five rows:")
print(fear_greed_data.tail().to_string())

Shape: (2644, 4)
Columns: ['timestamp', 'value', 'classification', 'date']
Total memory usage (bytes): 347383
                 dtype  memory_bytes
timestamp        int64         21152
value            int64         21152
classification  object        148951
date            object        155996

First five rows:
    timestamp  value classification        date
0  1517463000     30           Fear  2018-02-01
1  1517549400     15   Extreme Fear  2018-02-02
2  1517635800     40           Fear  2018-02-03
3  1517722200     24   Extreme Fear  2018-02-04
4  1517808600     11   Extreme Fear  2018-02-05

Last five rows:
       timestamp  value classification        date
2639  1745818200     54        Neutral  2025-04-28
2640  1745904600     60          Greed  2025-04-29
2641  1745991000     56          Greed  2025-04-30
2642  1746077400     53        Neutral  2025-05-01
2643  1746163800     67          Greed  2025-05-02


## Data Quality Assessment

In [6]:
def quality_table(summary: dict) -> pd.DataFrame:
    """Combine missing-value and cardinality counts by column."""
    return pd.DataFrame(
        {
            "missing_values": summary["missing_values"],
            "unique_values_including_missing": summary["unique_values"],
        }
    )

print("Historical duplicate rows:", historical_summary["duplicate_rows"])
print(quality_table(historical_summary).to_string())
print("\nFear & Greed duplicate rows:", fear_greed_summary["duplicate_rows"])
print(quality_table(fear_greed_summary).to_string())

Historical duplicate rows: 0
                  missing_values  unique_values_including_missing
Account                        0                               32
Coin                           0                              246
Execution Price                0                            60162
Size Tokens                    0                            59304
Size USD                       0                           118493
Side                           0                                2
Timestamp IST                  0                            27977
Start Position                 0                           196923
Direction                      0                               12
Closed PnL                     0                            90720
Transaction Hash               0                           101184
Order ID                       0                            50555
Crossed                        0                                2
Fee                            0               

### Numeric Summary Statistics

In [7]:
print("Historical numeric summary:")
print(historical_data.describe(include="number").T.to_string())
print("\nFear & Greed numeric summary:")
print(fear_greed_data.describe(include="number").T.to_string())

Historical numeric summary:
                    count          mean           std           min           25%           50%           75%           max
Execution Price  211224.0  1.141472e+04  2.944765e+04  4.530000e-06  4.854700e+00  1.828000e+01  1.015800e+02  1.090040e+05
Size Tokens      211224.0  4.623365e+03  1.042729e+05  8.740000e-07  2.940000e+00  3.200000e+01  1.879025e+02  1.582244e+07
Size USD         211224.0  5.639451e+03  3.657514e+04  0.000000e+00  1.937900e+02  5.970450e+02  2.058960e+03  3.921431e+06
Start Position   211224.0 -2.994625e+04  6.738074e+05 -1.433463e+07 -3.762311e+02  8.472793e+01  9.337278e+03  3.050948e+07
Closed PnL       211224.0  4.874900e+01  9.191648e+02 -1.179901e+05  0.000000e+00  0.000000e+00  5.792797e+00  1.353291e+05
Order ID         211224.0  6.965388e+10  1.835753e+10  1.732711e+08  5.983853e+10  7.442939e+10  8.335543e+10  9.014923e+10
Fee              211224.0  1.163967e+00  6.758854e+00 -1.175712e+00  1.612094e-02  8.957750e-02  3.93811

### Categorical Summaries

In [8]:
def categorical_summary(dataframe: pd.DataFrame) -> pd.DataFrame:
    """Describe text, category, and boolean columns without conversion."""
    categorical_columns = dataframe.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns
    return dataframe[categorical_columns].describe().T

print("Historical categorical summary:")
print(categorical_summary(historical_data).to_string())
print("\nFear & Greed categorical summary:")
print(categorical_summary(fear_greed_data).to_string())

Historical categorical summary:
                   count  unique                                                                 top    freq
Account           211224      32                          0xbee1707d6b44d4d52bfe19e41f8a828645437aab   40184
Coin              211224     246                                                                HYPE   68005
Side              211224       2                                                                SELL  108528
Timestamp IST     211224   27977                                                    14-02-2025 00:31     441
Direction         211224      12                                                           Open Long   49895
Transaction Hash  211224  101184  0x0000000000000000000000000000000000000000000000000000000000000000    9032
Crossed           211224       2                                                                True  128403

Fear & Greed categorical summary:
               count unique         top freq
classification  

## Date Validation

The following cells inspect stored values and inferred data types only. They deliberately do not parse or convert dates.

In [9]:
print("Historical Timestamp dtype:", historical_data["Timestamp"].dtype)
print("Historical Timestamp unique values:", historical_data["Timestamp"].nunique(dropna=False))
print(historical_data["Timestamp"].value_counts(dropna=False).sort_index().to_string())

print("\nHistorical Timestamp IST dtype:", historical_data["Timestamp IST"].dtype)
print("Historical Timestamp IST unique values:", historical_data["Timestamp IST"].nunique(dropna=False))
print("First five values:", historical_data["Timestamp IST"].head().tolist())
print("Last five values:", historical_data["Timestamp IST"].tail().tolist())

print("\nFear & Greed date dtype:", fear_greed_data["date"].dtype)
print("Fear & Greed date unique values:", fear_greed_data["date"].nunique(dropna=False))
print("First five values:", fear_greed_data["date"].head().tolist())
print("Last five values:", fear_greed_data["date"].tail().tolist())

Historical Timestamp dtype: float64
Historical Timestamp unique values: 7
Timestamp
1.680000e+12         3
1.700000e+12      1045
1.710000e+12      6962
1.720000e+12      7141
1.730000e+12     35241
1.740000e+12    133871
1.750000e+12     26961

Historical Timestamp IST dtype: object
Historical Timestamp IST unique values: 27977
First five values: ['02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50', '02-12-2024 22:50']
Last five values: ['25-04-2025 15:35', '25-04-2025 15:35', '25-04-2025 15:35', '25-04-2025 15:35', '25-04-2025 15:35']

Fear & Greed date dtype: object
Fear & Greed date unique values: 2644
First five values: ['2018-02-01', '2018-02-02', '2018-02-03', '2018-02-04', '2018-02-05']
Last five values: ['2025-04-28', '2025-04-29', '2025-04-30', '2025-05-01', '2025-05-02']


## Initial Observations

- The historical trader dataset contains **211,224 rows and 16 columns**; the Fear & Greed dataset contains **2,644 rows and 4 columns**.
- Pandas detects **no missing values and no fully duplicated rows** in either raw dataset.
- The trader dataset contains 32 accounts and 246 coins. Several fields have high cardinality, including `Start Position` (196,923 distinct values), `Fee` (138,802), `Size USD` (118,493), and `Transaction Hash` (101,184).
- `Timestamp IST` is loaded as text and has 27,977 distinct values. The numeric `Timestamp` field is loaded as `float64` but has only seven distinct values, so its precision and intended meaning require validation before it is used.
- The Fear & Greed `date` field is loaded as text, while its `timestamp` field is loaded as an integer. Each has 2,644 distinct values.
- A calendar date derived from `Timestamp IST` and the Fear & Greed `date` field appear to be potential future merge keys. Their formats, time zones, ranges, and overlap must be validated before any merge.
- Likely Phase 2 preprocessing requirements include explicit datetime parsing and validation of timestamp precision. No preprocessing has been performed in this phase.